## Setup

In [ ]:
!pip install -q trl bitsandbytes accelerate peft datasets transformers


## Authentication

In [ ]:
import os, torch
from huggingface_hub import login
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret('HF_TOKEN')
except Exception:
    hf_token = os.environ.get('HF_TOKEN', '')
login(hf_token)
print(f'GPUs: {torch.cuda.device_count()}')
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f'  GPU {i}: {p.name}  {p.total_memory/1e9:.1f} GB')


## 1 · Dataset Preparation

> **v9 scale-up:** 100 samples per label = **500 forget + 500 retain** (5 labels × 100 each). Stratified split ensures equal label distribution in both sets.

In [ ]:
import os
from collections import Counter
from datasets import load_dataset, concatenate_datasets

LABEL_COLUMN      = 'label'
SAMPLES_PER_LABEL = 200  # 100 → forget, 100 → retain (500 each for 5 labels)
SEED              = 42
DATA_DIR          = '/kaggle/working/data_splits'

print('[1] Loading dataset...')
full_dataset = load_dataset('Novaspree/factify_5K_enriched', split='train')
labels = full_dataset.unique(LABEL_COLUMN)
print(f'Labels ({len(labels)}): {labels}')

subset_list = []
for label in labels:
    ls = full_dataset.filter(lambda x: x[LABEL_COLUMN] == label).select(range(SAMPLES_PER_LABEL))
    subset_list.append(ls)

finetune_dataset = concatenate_datasets(subset_list).shuffle(seed=SEED)
finetune_dataset = finetune_dataset.class_encode_column(LABEL_COLUMN)

# 500 forget (100/label) + 500 retain (100/label)
split = finetune_dataset.train_test_split(
    test_size=500, stratify_by_column=LABEL_COLUMN, seed=SEED
)
forget_dataset = split['test']    # 500 samples — to be unlearned
retain_dataset = split['train']   # 500 samples — to be preserved

def format_for_training(example):
    return {'text': (
        '<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\n'
        f"{example['question']}<|eot_id|>"
        '<|start_header_id|>assistant<|end_header_id|>\n\n'
        f"{example['answer']}<|eot_id|>"
    )}

finetune_dataset = finetune_dataset.map(format_for_training)
forget_dataset   = forget_dataset.map(format_for_training)
retain_dataset   = retain_dataset.map(format_for_training)

os.makedirs(DATA_DIR, exist_ok=True)
finetune_dataset.save_to_disk(f'{DATA_DIR}/finetune_dataset')
forget_dataset.save_to_disk(f'{DATA_DIR}/forget_dataset')
retain_dataset.save_to_disk(f'{DATA_DIR}/retain_dataset')

print(f'Forget : {len(forget_dataset)} | dist: {dict(Counter(forget_dataset[LABEL_COLUMN]))}')
print(f'Retain : {len(retain_dataset)} | dist: {dict(Counter(retain_dataset[LABEL_COLUMN]))}')


## 2 · Mid-Layer LoRA Fine-Tuning

> LoRA targets layers 7–20 across all projection types (`down_proj`, `up_proj`, `q_proj`, `k_proj`, `v_proj`, `o_proj`, `gate_proj`).  
> **v8 change:** adapter is saved but **NOT merged** here — merging happens in Cell 6 after unlearning, so we can operate in adapter space.

In [ ]:
import torch
from transformers import (AutoTokenizer, AutoModelForCausalLM,
                          TrainingArguments, BitsAndBytesConfig)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
from datasets import load_from_disk

MODEL_NAME        = 'meta-llama/Llama-3.2-3B'
LORA_ADAPTER_PATH = '/kaggle/working/lora_adapter'   # adapter saved here (NOT merged)
DATA_DIR          = '/kaggle/working/data_splits'
MID_LAYER_START   = 7
MID_LAYER_END     = 20
MID_LAYERS        = list(range(MID_LAYER_START, MID_LAYER_END + 1))
LORA_RANK         = 32

finetune_dataset = load_from_disk(f'{DATA_DIR}/finetune_dataset')

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True,
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb_config, device_map='auto'
)
base_model = prepare_model_for_kbit_training(base_model)

lora_config = LoraConfig(
    r=LORA_RANK, lora_alpha=LORA_RANK * 2,
    target_modules=['gate_proj','down_proj','up_proj','q_proj','k_proj','v_proj','o_proj'],
    layers_to_transform=MID_LAYERS,
    lora_dropout=0.05, bias='none', task_type='CAUSAL_LM',
)
peft_model = get_peft_model(base_model, lora_config)
peft_model.print_trainable_parameters()

training_args = TrainingArguments(
    output_dir='/kaggle/working/lora_results',
    per_device_train_batch_size=4, gradient_accumulation_steps=4,
    learning_rate=2e-4, num_train_epochs=10, warmup_ratio=0.05,
    logging_steps=10, save_strategy='no',
    bf16=True, fp16=False, report_to='none',
    gradient_checkpointing=True, ddp_find_unused_parameters=False,
)
trainer = SFTTrainer(model=peft_model, train_dataset=finetune_dataset, args=training_args)
trainer.train()

# Save adapter only — do NOT merge yet (v8 unlearns in adapter space)
trainer.model.save_pretrained(LORA_ADAPTER_PATH)
tokenizer.save_pretrained(LORA_ADAPTER_PATH)
print(f'LoRA adapter saved → {LORA_ADAPTER_PATH}  (merge happens AFTER unlearning)')


## 3 · RecursiveMAAT v8.1 — Retain-Repaired LoRA-Native Engine

> **Three targeted fixes for retain performance (forget quality unchanged):**
> - **FIX 1 — KL reference:** `KL(current ‖ finetuned_adapter)` instead of `KL(current ‖ base_model)`.  The base model does not know the retain facts; anchoring to it pulled the adapter away from correct answers.
> - **FIX 2 — SVD pruning scope:** MLP-only (`down_proj`, `up_proj`, `gate_proj`), ratio 30%→15%.  Pruning `q_proj`/`v_proj` destroyed instruction-following, causing "The following is a list of..." hallucinations.
> - **FIX 3 — Retain repair:** 40→100 steps, scope expanded to all 7 LoRA module types (`gate_proj`, `down_proj`, `up_proj`, `q_proj`, `k_proj`, `v_proj`, `o_proj`).  Previously `gate_proj`/`k_proj`/`o_proj` were never repaired.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm
import re, gc, random


def get_input_device(model):
    return next(model.parameters()).device


def format_prompt_eval(question):
    return (
        '<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\n'
        f'{question}<|eot_id|>'
        '<|start_header_id|>assistant<|end_header_id|>\n\n'
    )


def make_answer_only_labels(input_ids, prompt_len):
    labels = input_ids.clone()
    labels[:, :prompt_len] = -100
    return labels


def clean_output(text: str) -> str:
    import re
    # Escape the brackets \[ \] and the curly braces \{ \}
    # This prevents the "unbalanced parenthesis" error
    pattern = r'\s*(CLIIIK|\?>|\];|\]|/|/\*|<!|\{\{|\}\}).*$'
    
    # Alternatively, a safer way to match those specific artifacts:
    text = re.sub(r'\s*(CLIIIK|\?>|\];|\]|/\*|<!|\{\{|\}\}).*$', '', text, flags=re.DOTALL)
    return text.strip()


class RecursiveMAAT_v8:
    """
    LoRA-native unlearning (v8.1 — retain-repaired).

    Changes vs v8.0:
      FIX 1 – KL reference = finetuned adapter (initial_weights saved before
               unlearning), NOT the base model.  The base model does not know
               the retain facts; using it as the KL target pulled the adapter
               away from correct retain answers.  Using the pre-unlearn adapter
               as reference keeps retain outputs anchored to what the model
               already learned.

      FIX 2 – SVD pruning is now MLP-only (down_proj, up_proj, gate_proj).
               Pruning q_proj / v_proj rank dims at 30% destroyed the model's
               instruction-following pathway, causing the "following is a list"
               hallucination pattern.  Attention modules are excluded from
               pruning; prune ratio reduced to 15%.

      FIX 3 – Retain repair scope expanded to ALL 7 LoRA module types
               (gate_proj, down_proj, up_proj, q_proj, k_proj, v_proj, o_proj)
               and steps increased 40→100.  The original repair only touched 4
               types, leaving gate_proj / k_proj / o_proj in a degraded state.

    Unlearn_step (GradProject) and Fisher logic are unchanged — forgetting
    quality was already good.
    """

    # Modules touched by unlearning (forget gradient + GradProject)
    UNLEARN_MODULE_TYPES = ('down_proj', 'up_proj', 'q_proj', 'v_proj')

    # Modules repaired in retain_repair (all LoRA targets from fine-tuning)
    REPAIR_MODULE_TYPES  = ('down_proj', 'up_proj', 'gate_proj',
                            'q_proj', 'k_proj', 'v_proj', 'o_proj')

    # SVD pruning only on MLP (attention excluded — see FIX 2)
    SVD_MODULE_TYPES     = ('down_proj', 'up_proj', 'gate_proj')

    def __init__(
        self,
        model,
        mid_layer_start  = 7,
        mid_layer_end    = 20,
        learning_rate    = 1e-3,
        max_grad_norm    = 1.0,
        kl_temp          = 0.7,
        do_svd_prune     = True,
        svd_prune_ratio  = 0.15,    # FIX 2: 30% → 15%, MLP-only
    ):
        self.model          = model
        self.lr             = learning_rate
        self.max_grad_norm  = max_grad_norm
        self.kl_temp        = kl_temp
        self.do_svd_prune   = do_svd_prune
        self.svd_prune_ratio = svd_prune_ratio

        # ── Discover unlearn-target adapter modules ──────────────────
        self.lora_a_modules = []
        self.lora_b_modules = []

        for name, mod in model.named_modules():
            if not isinstance(mod, nn.Linear): continue
            m = re.search(r'layers\.(\d+)', name)
            if not m: continue
            if not (mid_layer_start <= int(m.group(1)) <= mid_layer_end): continue
            if not any(mt in name for mt in self.UNLEARN_MODULE_TYPES): continue
            if 'lora_A' in name:
                self.lora_a_modules.append((name, mod))
            elif 'lora_B' in name:
                self.lora_b_modules.append((name, mod))

        self.target_modules = self.lora_a_modules + self.lora_b_modules

        if not self.target_modules:
            raise RuntimeError(
                "No lora_A/lora_B modules found. "
                "Pass a PeftModel with is_trainable=True."
            )

        # ── FIX 1: save finetuned adapter as KL reference ────────────
        # These are the pre-unlearn adapter weights.  Used in both
        # unlearn_step (KL retain loss) and retain_repair (KL target).
        self.finetuned_weights = {}
        for name, mod in model.named_modules():
            if not isinstance(mod, nn.Linear): continue
            if 'lora_A' in name or 'lora_B' in name:
                self.finetuned_weights[name] = mod.weight.data.clone().cpu()

        # ── Freeze base; unfreeze only unlearn-target adapter weights ──
        for p in model.parameters():
            p.requires_grad = False
        for _, mod in self.target_modules:
            mod.weight.requires_grad = True

        total  = sum(m.weight.numel() for _, m in self.target_modules)
        all_p  = sum(p.numel() for p in model.parameters())
        print(f'RecursiveMAAT v8.1 ready')
        print(f'  Unlearn modules  : {len(self.target_modules)} '
              f'({len(self.lora_a_modules)} A + {len(self.lora_b_modules)} B)')
        print(f'  Trainable params : {total:,} / {all_p:,} '
              f'({100*total/all_p:.3f}%)')
        print(f'  lr={learning_rate} | kl_temp={kl_temp} | '
              f'svd_prune={do_svd_prune} ({svd_prune_ratio:.0%}, MLP-only)')

    # ─────────────────────────────────────────────────────────────────
    def _encode(self, question, answer, tokenizer, device):
        prompt = format_prompt_eval(question)
        full   = prompt + answer
        plen   = tokenizer(prompt, return_tensors='pt').input_ids.shape[1]
        enc    = tokenizer(full, return_tensors='pt',
                           truncation=True, max_length=512).to(device)
        labels = make_answer_only_labels(enc['input_ids'], plen)
        return enc, labels

    def _encode_prompt_only(self, question, tokenizer, device):
        prompt = format_prompt_eval(question)
        enc    = tokenizer(prompt, return_tensors='pt',
                           truncation=True, max_length=512).to(device)
        return enc

    # ─────────────────────────────────────────────────────────────────
    def _get_finetuned_logits(self, enc):
        """
        FIX 1: Forward pass through the FINETUNED (pre-unlearn) adapter.
        Temporarily swaps in the saved finetuned_weights, runs forward,
        then restores the current (partially unlearned) weights.
        """
        self.model.eval()
        current = {n: mod.weight.data.clone()
                   for n, mod in self.target_modules}
        for n, mod in self.target_modules:
            saved = self.finetuned_weights.get(n)
            if saved is not None:
                mod.weight.data.copy_(saved.to(mod.weight.device))
        with torch.no_grad():
            logits = self.model(**enc).logits.float().detach()
        for n, mod in self.target_modules:
            mod.weight.data.copy_(current[n])
        del current
        return logits

    # ─────────────────────────────────────────────────────────────────
    def unlearn_step(
        self,
        forget_question, forget_answer,
        retain_pool, retain_start_idx,
        tokenizer, steps=15,
    ):
        """
        GradProject unlearning — unchanged from v8.0 (forgetting is good).
        KL retain loss now uses finetuned adapter as reference (FIX 1).
        """
        self.model.train()
        device    = get_input_device(self.model)
        forget_enc, forget_labels = self._encode(
            forget_question, forget_answer, tokenizer, device
        )
        optimizer = torch.optim.Adam(
            [mod.weight for _, mod in self.target_modules], lr=self.lr
        )

        for step in range(steps):
            rs = retain_pool[(retain_start_idx + step) % len(retain_pool)]
            retain_enc, _ = self._encode(rs['question'], rs['answer'],
                                         tokenizer, device)

            # A: forget gradient
            optimizer.zero_grad()
            forget_loss = self.model(**forget_enc, labels=forget_labels).loss
            forget_loss.backward()
            forget_grads = {
                n: mod.weight.grad.clone()
                for n, mod in self.target_modules
                if mod.weight.grad is not None
            }

            # B: KL retain gradient — ref = finetuned adapter (FIX 1)
            optimizer.zero_grad()
            ref_logits  = self._get_finetuned_logits(retain_enc)   # FIX 1
            ref_probs   = F.softmax(ref_logits / self.kl_temp, dim=-1)
            del ref_logits

            self.model.train()
            model_logits = self.model(**retain_enc).logits.float()
            kl_loss = F.kl_div(
                F.log_softmax(model_logits / self.kl_temp, dim=-1),
                ref_probs, reduction='batchmean'
            )
            del ref_probs, model_logits
            kl_loss.backward()

            # C: GradProject — orthogonalise forget grad w.r.t. retain grad
            with torch.no_grad():
                for name, mod in self.target_modules:
                    g_r = mod.weight.grad
                    if g_r is None:
                        continue
                    g_f = forget_grads.get(name)
                    if g_f is None:
                        mod.weight.grad.zero_()
                        continue
                    g_f     = g_f.to(g_r.device)
                    dot     = (g_f * g_r).sum()
                    g_r_sq  = (g_r * g_r).sum().clamp(min=1e-12)
                    g_f_proj = g_f - (dot / g_r_sq) * g_r
                    mod.weight.grad.copy_(-g_f_proj)   # ascent

            torch.nn.utils.clip_grad_norm_(
                [mod.weight for _, mod in self.target_modules],
                self.max_grad_norm
            )
            optimizer.step()
            del forget_grads, retain_enc

        torch.cuda.empty_cache()

    # ─────────────────────────────────────────────────────────────────
    def svd_prune(self, forget_dataset, tokenizer, n_score_samples=20):
        """
        FIX 2: Prune only MLP modules (down_proj, up_proj, gate_proj).
        Attention modules (q_proj, v_proj) are excluded to preserve
        instruction-following capability.  Prune ratio 15% (was 30%).
        """
        if not self.do_svd_prune:
            return

        print(f'\nSVD rank pruning (MLP-only, {self.svd_prune_ratio:.0%})...')
        device = get_input_device(self.model)
        self.model.eval()

        def base_key(name):
            return re.sub(r'\.lora_[AB].*', '', name)

        # Build pairs — MLP-only (FIX 2)
        a_by_key = {}
        b_by_key = {}
        for name, mod in self.lora_a_modules:
            if any(mt in name for mt in self.SVD_MODULE_TYPES):
                a_by_key[base_key(name)] = (name, mod)
        for name, mod in self.lora_b_modules:
            if any(mt in name for mt in self.SVD_MODULE_TYPES):
                b_by_key[base_key(name)] = (name, mod)
        common_keys = set(a_by_key) & set(b_by_key)
        print(f'  Pruning {len(common_keys)} MLP layer pairs (attention skipped)')

        rank_signal = {}
        n_samples   = min(n_score_samples, len(forget_dataset))

        for i in tqdm(range(n_samples), desc='Scoring rank dims'):
            sample = forget_dataset[i]
            enc, labels = self._encode(
                sample['question'], sample['answer'], tokenizer, device
            )
            self.model.zero_grad()
            self.model(**enc, labels=labels).loss.backward()

            with torch.no_grad():
                for key in common_keys:
                    _, mod_b = b_by_key[key]
                    if mod_b.weight.grad is None: continue
                    col_norms = mod_b.weight.grad.float().norm(dim=0).cpu()
                    rank_signal[key] = rank_signal.get(key, 0) + col_norms

        self.model.zero_grad()
        gc.collect(); torch.cuda.empty_cache()

        total_zeroed = 0
        for key in common_keys:
            _, mod_a = a_by_key[key]
            _, mod_b = b_by_key[key]
            rank   = mod_a.weight.shape[0]
            scores = rank_signal.get(key, torch.ones(rank))
            n_prune = max(1, int(self.svd_prune_ratio * rank))
            prune_dims = scores.argsort(descending=True)[:n_prune]
            with torch.no_grad():
                mod_b.weight.data[:, prune_dims] = 0.0
                mod_a.weight.data[prune_dims, :] = 0.0
            total_zeroed += n_prune

        print(f'Rank dims zeroed: {total_zeroed} across {len(common_keys)} MLP pairs.')

    # ─────────────────────────────────────────────────────────────────
    def retain_repair(self, retain_dataset, tokenizer,
                      n_steps=100, repair_lr=5e-5):
        """
        FIX 3: Expanded scope (all 7 LoRA module types) + longer 100 steps.
        KL reference = finetuned adapter (FIX 1, same as unlearn_step).

        Previously: 40 steps on 4 module types only — gate_proj, k_proj,
        and o_proj were left in a degraded state, causing confusion/uncertainty
        on retain answers.
        """
        print(f'\nRetain repair: {n_steps} steps, lr={repair_lr}')
        print(f'  Scope: {self.REPAIR_MODULE_TYPES}')
        device = get_input_device(self.model)
        self.model.train()

        # Collect ALL LoRA modules for repair (FIX 3 — broader scope)
        repair_modules = []
        for name, mod in self.model.named_modules():
            if not isinstance(mod, nn.Linear): continue
            m = re.search(r'layers\.(\d+)', name)
            if not m: continue
            layer_idx = int(m.group(1))
            # Repair all layers where LoRA was applied (7–20)
            if not (7 <= layer_idx <= 20): continue
            if not any(mt in name for mt in self.REPAIR_MODULE_TYPES): continue
            if 'lora_A' in name or 'lora_B' in name:
                mod.weight.requires_grad = True
                repair_modules.append((name, mod))

        print(f'  Repair modules: {len(repair_modules)}')

        optimizer = torch.optim.Adam(
            [mod.weight for _, mod in repair_modules], lr=repair_lr
        )
        samples = [retain_dataset[i] for i in range(len(retain_dataset))]
        random.shuffle(samples)

        for step in range(n_steps):
            sample = samples[step % len(samples)]
            enc, _ = self._encode(sample['question'], sample['answer'],
                                  tokenizer, device)

            optimizer.zero_grad()

            # KL vs finetuned adapter — FIX 1 + FIX 3
            ref_logits  = self._get_finetuned_logits(enc)
            ref_probs   = F.softmax(ref_logits / self.kl_temp, dim=-1)
            del ref_logits

            self.model.train()
            model_logits = self.model(**enc).logits.float()
            kl_loss = F.kl_div(
                F.log_softmax(model_logits / self.kl_temp, dim=-1),
                ref_probs, reduction='batchmean'
            )
            del ref_probs, model_logits
            kl_loss.backward()
            optimizer.step()

            if (step + 1) % 20 == 0:
                torch.cuda.empty_cache()
                print(f'  step {step+1}/{n_steps}  kl={kl_loss.item():.4f}')

        print('Retain repair done.')


print('RecursiveMAAT_v8_1, format_prompt_eval, get_input_device, '
      'make_answer_only_labels, clean_output — loaded.')


## 4 · Run MA'AT v8.1 Unlearning (Three-Phase) — 500-sample scale

> **Phase 1** — GradProject ascent on forget set (500 samples, unchanged algorithm)  
> **Phase 2** — SVD rank-dim pruning, **MLP-only**, ratio 15%, scored on 100 samples (FIX 2)  
> **Phase 3** — KL retain repair vs finetuned adapter, **200 steps**, all 7 module types (FIX 1 + FIX 3)

In [ ]:
import os, shutil, gc, random
import torch
from peft import PeftModel
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_from_disk

MODEL_NAME        = 'meta-llama/Llama-3.2-3B'
LORA_ADAPTER_PATH = '/kaggle/working/lora_adapter'
UNLEARNED_ADAPTER = '/kaggle/working/lora_adapter_unlearned'
DATA_DIR          = '/kaggle/working/data_splits'

# ── Config ───────────────────────────────────────────────────────────
MID_LAYER_START  = 7
MID_LAYER_END    = 20
LEARNING_RATE    = 1e-5
MAX_GRAD_NORM    = 1.0
UNLEARN_STEPS    = 15
KL_TEMP          = 0.7

# FIX 2: reduced ratio, MLP-only (handled inside engine)
DO_SVD_PRUNE     = True
SVD_PRUNE_RATIO  = 0.15
SVD_SCORE_SAMPLES = 100  # scaled: score 100 of 500 forget samples

# FIX 3: 40 → 200 steps (scaled for 500-sample retain set), all 7 module types
DO_RETAIN_REPAIR = True
REPAIR_STEPS     = 200
REPAIR_LR        = 5e-5
# ─────────────────────────────────────────────────────────────────────

gc.collect(); torch.cuda.empty_cache()
random.seed(42)

forget_dataset = load_from_disk(f'{DATA_DIR}/forget_dataset')
retain_dataset = load_from_disk(f'{DATA_DIR}/retain_dataset')
retain_list    = [retain_dataset[i] for i in range(len(retain_dataset))]
random.shuffle(retain_list)
print(f'Forget: {len(forget_dataset)} | Retain: {len(retain_dataset)}')

tokenizer = AutoTokenizer.from_pretrained(LORA_ADAPTER_PATH)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print('Loading base model + LoRA adapter as PeftModel...')
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float16, device_map='auto'
)
model = PeftModel.from_pretrained(
    base_model, LORA_ADAPTER_PATH, is_trainable=True
)
model.eval()

# Instantiate v8.1 engine
# FIX 1 is automatic: __init__ saves finetuned_weights before any unlearning
maat = RecursiveMAAT_v8(
    model,
    mid_layer_start  = MID_LAYER_START,
    mid_layer_end    = MID_LAYER_END,
    learning_rate    = LEARNING_RATE,
    max_grad_norm    = MAX_GRAD_NORM,
    kl_temp          = KL_TEMP,
    do_svd_prune     = DO_SVD_PRUNE,
    svd_prune_ratio  = SVD_PRUNE_RATIO,
)

# ── Phase 1: GradProject unlearning ──────────────────────────────────
print(f'\n[Phase 1] GradProject unlearning {len(forget_dataset)} samples '
      f'({UNLEARN_STEPS} steps each)...')

for i, forget_sample in enumerate(forget_dataset):
    maat.unlearn_step(
        forget_question  = forget_sample['question'],
        forget_answer    = forget_sample['answer'],
        retain_pool      = retain_list,
        retain_start_idx = i,
        tokenizer        = tokenizer,
        steps            = UNLEARN_STEPS,
    )
    if (i + 1) % 5 == 0:
        torch.cuda.empty_cache()
        print(f'  [{i+1}/{len(forget_dataset)}] done')
    if (i + 1) % 50 == 0:
        ckpt_path = f'/kaggle/working/lora_adapter_ckpt_{i+1}'
        model.save_pretrained(ckpt_path)
        print(f'  Checkpoint saved → {ckpt_path}')

# ── Phase 2: SVD rank pruning (MLP-only, 15%) ────────────────────────
if DO_SVD_PRUNE:
    print('\n[Phase 2] SVD rank pruning (MLP-only)...')
    maat.svd_prune(forget_dataset, tokenizer,
                   n_score_samples=SVD_SCORE_SAMPLES)

# ── Phase 3: Retain repair (100 steps, all 7 module types) ───────────
if DO_RETAIN_REPAIR:
    print('\n[Phase 3] Retain repair (expanded scope)...')
    maat.retain_repair(retain_dataset, tokenizer,
                       n_steps=REPAIR_STEPS, repair_lr=REPAIR_LR)

# Save unlearned adapter
if os.path.exists(UNLEARNED_ADAPTER): shutil.rmtree(UNLEARNED_ADAPTER)
model.save_pretrained(UNLEARNED_ADAPTER)
tokenizer.save_pretrained(UNLEARNED_ADAPTER)
print(f'\nUnlearned adapter → {UNLEARNED_ADAPTER}')


## 5 · Merge Unlearned Adapter → Full Model

> Merge happens here (after unlearning), producing a standard HF model ready for inference and evaluation.

In [ ]:
import gc, shutil
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM

MODEL_NAME        = 'meta-llama/Llama-3.2-3B'
UNLEARNED_ADAPTER = '/kaggle/working/lora_adapter_unlearned'
MERGED_MODEL_PATH = '/kaggle/working/Llama-3.2-3B-Unlearned-Merged'

# Free unlearning model from GPU before reloading for merge
try:
    del model, maat, base_model
except NameError:
    pass
gc.collect(); torch.cuda.empty_cache()

print('Merging unlearned adapter → full model...')
base_for_merge = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float16, device_map='auto'
)
merged = PeftModel.from_pretrained(base_for_merge, UNLEARNED_ADAPTER).merge_and_unload()

if os.path.exists(MERGED_MODEL_PATH): shutil.rmtree(MERGED_MODEL_PATH)
merged.save_pretrained(MERGED_MODEL_PATH)
tokenizer.save_pretrained(MERGED_MODEL_PATH)
print(f'Merged unlearned model → {MERGED_MODEL_PATH}')

del merged, base_for_merge
gc.collect(); torch.cuda.empty_cache()


## 6 · Generate & Save Answers (for LLM-as-Judge)

In [ ]:
import json, os, gc, re
import torch
from datasets import load_from_disk
from transformers import AutoTokenizer, AutoModelForCausalLM

MERGED_MODEL_PATH = '/kaggle/working/Llama-3.2-3B-Unlearned-Merged'
DATA_DIR          = '/kaggle/working/data_splits'
OUT_DIR           = '/kaggle/working/eval_inputs'
MAX_NEW_TOKENS    = 100

os.makedirs(OUT_DIR, exist_ok=True)
gc.collect(); torch.cuda.empty_cache()

tokenizer = AutoTokenizer.from_pretrained(MERGED_MODEL_PATH)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MERGED_MODEL_PATH, torch_dtype=torch.float16, device_map='auto'
)
model.eval()
input_device = get_input_device(model)
torch.manual_seed(42)

eos_ids = list({tokenizer.eos_token_id,
                tokenizer.convert_tokens_to_ids('<|eot_id|>')})
eos_ids = [e for e in eos_ids if e is not None and e >= 0]


def generate_answer(question: str) -> str:
    prompt  = format_prompt_eval(question)
    inputs  = tokenizer(prompt, return_tensors='pt',
                        truncation=True, max_length=512).to(input_device)
    with torch.no_grad():
        gen = model.generate(
            **inputs,
            max_new_tokens      = MAX_NEW_TOKENS,
            do_sample           = False,
            repetition_penalty  = 1.3,
            eos_token_id        = eos_ids,
            pad_token_id        = tokenizer.eos_token_id,
        )
    new_ids = gen[0][inputs['input_ids'].shape[1]:]
    raw     = tokenizer.decode(new_ids, skip_special_tokens=True).strip()
    return clean_output(raw)


def collect_answers(dataset, label_feature, split_name: str) -> list:
    records = []
    print(f'[{split_name}] {len(dataset)} samples...')
    for i, sample in enumerate(dataset):
        records.append({
            'split'        : split_name,
            'idx'          : i,
            'label'        : label_feature.int2str(sample['label']),
            'question'     : sample['question'],
            'ground_truth' : sample['answer'],
            'model_answer' : generate_answer(sample['question']),
        })
        if (i + 1) % 10 == 0:
            torch.cuda.empty_cache()
            print(f'  [{i+1}/{len(dataset)}] done')
    return records


forget_dataset = load_from_disk(f'{DATA_DIR}/forget_dataset')
retain_dataset = load_from_disk(f'{DATA_DIR}/retain_dataset')
label_feature  = forget_dataset.features['label']

forget_records = collect_answers(forget_dataset, label_feature, 'forget')
retain_records = collect_answers(retain_dataset, label_feature, 'retain')
all_records    = forget_records + retain_records

for fname, records in [
    ('forget_answers.json', forget_records),
    ('retain_answers.json', retain_records),
    ('all_answers.json',    all_records),
]:
    with open(os.path.join(OUT_DIR, fname), 'w') as f:
        json.dump(records, f, indent=2, ensure_ascii=False)

print(f'\nSaved to {OUT_DIR}/')
for fname in ['forget_answers.json', 'retain_answers.json', 'all_answers.json']:
    p = os.path.join(OUT_DIR, fname)
    print(f'  {p}  ({os.path.getsize(p)/1024:.1f} KB)')
print('\n── Sample forget ──')
print(json.dumps(forget_records[0], indent=2))
print('\n── Sample retain ──')
print(json.dumps(retain_records[0], indent=2))
